# Model Evaluation & Validation

Welcome to the twelfth notebook in our repository. This notebook focuses exclusively on how to properly evaluate, compare, validate, and select Machine Learning models.

==================================================

# 1. Why Model Evaluation Matters

==================================================

**Why training a model is not enough**
Simply memorizing training data does not make a model useful. The true test of a machine learning model is its performance on data it has NEVER seen.

**Why unseen data matters**
We build models to predict the future. Since the future is unseen data, evaluating on unseen current data is the only sensible proxy.

**What "good model" means**
A good model captures the underlying signal and ignores the noise.

**Generalization**
How well the concepts learned by a model apply to specific examples not seen by the model. 

**Model evaluation vs model training**
- Training applies optimization to minimize errors.
- Evaluation represents the unbiased testing phase using strictly held-out data to score real performance.

*Real-world example:* A student memorizes past exams (training) and scores 100%. In the real exam (unseen data) with entirely new questions, they fail. The student failed to *generalize*.


==================================================

# 2. Training vs Validation vs Test Data

==================================================

- **Training set:** The largest chunk of data. The model actively learns patterns from this.
- **Validation set:** Used during model tuning (e.g., hyperparameters). Used to safely compare models without touching the final test data.
- **Test set:** The final exam. Used strictly ONCE at the end.

Dataset $\rightarrow$ Training $\rightarrow$ Validation $\rightarrow$ Final Test

**Why looking at the test set repeatedly leads to overfitting**
If you evaluate multiple models on the test set and pick the best, you "leaked" information about the test set into your decision process.


==================================================

# 3. Train-Test Split

==================================================

We normally use Scikit-learn to split data.


In [ ]:
from sklearn.model_selection import train_test_split


**Key parameters:**
- `test_size`: 0.2 means 20% test data.
- `random_state`: Sets the random seed.
- `shuffle`: Randomizes order.
- `stratify`: Ensures class proportions remain identical in train and test (essential for imbalanced classification).


In [ ]:
import numpy as np
import pandas as pd

# Classification mock data
X_cls = np.random.randn(100, 5)
y_cls = np.array([0]*90 + [1]*10)
X_tr, X_ts, y_tr, y_ts = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# Regression mock data
X_reg = np.random.randn(100, 5)
y_reg = np.random.randn(100)
X_r_tr, X_r_ts, y_r_tr, y_r_ts = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)


==================================================

# 4. Classification Evaluation

==================================================

### Accuracy
$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$
- **Useful when:** Classes are balanced.
- **Misleading when:** Classes are imbalanced.

### Precision
$$Precision = \frac{TP}{TP + FP}$$
- **Useful when:** False positives are expensive (e.g., spam filter deleting important emails).

### Recall
$$Recall = \frac{TP}{TP + FN}$$
- **Useful when:** False negatives are expensive (e.g., cancer screening).

### F1 Score
$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$
- **Useful when:** You want a balance between Precision and Recall.


==================================================

# 5. Confusion Matrix

==================================================

- **True Positive (TP):** We predicted Positive, and it IS Positive.
- **True Negative (TN):** We predicted Negative, and it IS Negative.
- **False Positive (FP):** We predicted Positive, but it IS Negative (Type I Error).
- **False Negative (FN):** We predicted Negative, but it IS Positive (Type II Error).


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
X_disease = np.random.randn(200, 2)
y_disease = np.array([0]*160 + [1]*40)

X_tr_d, X_ts_d, y_tr_d, y_ts_d = train_test_split(X_disease, y_disease, test_size=0.3, random_state=42)
model_disease = LogisticRegression().fit(X_tr_d, y_tr_d)
preds = model_disease.predict(X_ts_d)

cm = confusion_matrix(y_ts_d, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Healthy', 'Disease'])
disp.plot(cmap='Blues')
plt.title('Disease Detection Confusion Matrix')
plt.show()


**Quadrants:**
- Top-Left (TN): Healthy people properly predicted as healthy.
- Bottom-Right (TP): Sick people correctly identified.
- Top-Right (FP): Healthy people improperly flagged as sick.
- Bottom-Left (FN): Sick people told they are healthy.


==================================================

# 6. Precision-Recall Trade-off

==================================================

- **Why increasing recall can reduce precision:** Flagging more cases as positive ensures you find everyone, but you flag healthy people too.
- **Why increasing precision can reduce recall:** Demanding perfect certainty for positive labels guarantees you miss subtle true positives.
- **Threshold-based classification:** Classifiers output probabilities. We use an arbitrary threshold (e.g., > 0.5) to decide.


In [ ]:
plt.figure(figsize=(6, 4))
x = np.linspace(0, 1, 100)
plt.plot(x, -0.8 * x + 0.9, label="Recall", color='blue')
plt.plot(x, 0.8 * x + 0.2, label="Precision", color='green')
plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold')
plt.title('Precision-Recall Tradeoff')
plt.xlabel('Threshold')
plt.legend()
plt.show()


==================================================

# 7. Classification Threshold

==================================================

- **Default threshold = 0.5:** Usually standard, but inappropriate for severe disease tracking.
- **Lower threshold:** Catches more positives (Higher Recall), lowers Precision.
- **Higher threshold:** Demands certainty (Higher Precision), lowers Recall.


In [ ]:
probs = model_disease.predict_proba(X_ts_d)[:, 1]
print("0.5 predictions:", (probs > 0.5).astype(int)[:10])
print("0.3 threshold:", (probs > 0.3).astype(int)[:10])
print("0.8 threshold:", (probs > 0.8).astype(int)[:10])


==================================================

# 8. ROC Curve and AUC

==================================================

ROC Curve evaluates classification globally across all thresholds.
- **TPR:** True Positive Rate
- **FPR:** False Positive Rate
- **AUC:** Area Under Curve


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score


In [ ]:
fpr, tpr, thr = roc_curve(y_ts_d, probs)
auc_val = roc_auc_score(y_ts_d, probs)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f'ROC (AUC = {auc_val:.2f})', color='orange')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curve')
plt.legend(); plt.show()


- AUC close to 1: Strong discrimination.
- AUC around 0.5: Random-like performance.
Note: ROC-AUC can overestimate capabilities on heavily imbalanced problems.


==================================================

# 9. Precision-Recall Curve

==================================================

Precision-Recall Curves are immensely useful for imbalanced datasets because they ignore True Negatives.


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score


In [ ]:
prec, rec, thr_pr = precision_recall_curve(y_ts_d, probs)
pr_auc = average_precision_score(y_ts_d, probs)

plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f'PR (AUC = {pr_auc:.2f})', color='purple')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('PR Curve')
plt.legend(); plt.show()


==================================================

# 10. Regression Evaluation

==================================================

### MAE (Mean Absolute Error)
- **Formula:** Average absolute errors.
- **Intuition:** "On average, prediction is off by X units."
- **When useful:** Requires strict linear interpretability, robust to massive outliers.

### MSE (Mean Squared Error)
- **Formula:** Average squared errors.
- **Intuition:** Blows up errors squared.
- **When useful:** You want to brutally penalize enormous predictive misses.

### RMSE (Root Mean Squared Error)
- **Formula:** Square root of MSE.
- **Intuition:** Squares the penalties, but roots back to original target units.
- **When useful:** Industry default. Balances scaled units and outlier penalties.

### R² (Coefficient of Determination)
- **Formula:** Proportion of variance in dependent variable predictable from independent variable.
- **Intuition:** Goodness of relative fit.
- **When useful:** General scale-independent evaluations.


==================================================

# 11. MAE vs MSE vs RMSE

==================================================
| Metric | Outlier Sensitivity | Native Unit |
| :--- | :--- | :--- |
| **MAE** | Low | Yes |
| **MSE** | High | No |
| **RMSE**| High | Yes |


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_true = np.array([10, 20, 30, 40])
y_pred = np.array([12, 18, 30, 80])

print(f"MAE: {mean_absolute_error(y_true, y_pred)}")
print(f"MSE: {mean_squared_error(y_true, y_pred)}")
print(f"RMSE: {mean_squared_error(y_true, y_pred, squared=False)}")


==================================================

# 12. R² Score

==================================================
- **What R² measures:** Variance explained.
- **Interpretation:** Not "percentage accuracy."
- **R² = 1:** Perfect fit.
- **R² = 0:** Same as predicting the mean exactly every time.
- **Negative R²:** The model is worse than a flat line predicting the mean.


==================================================

# 13. Cross Validation

==================================================
Why is one split unreliable? Bad random draws exist.

**K-Fold Cross Validation:**
1. Split data into K folds.
2. Train on K-1 folds.
3. Validate on remaining fold.
4. Repeat K times.
5. Average results.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score


In [ ]:
from sklearn.tree import DecisionTreeClassifier
model_cv = DecisionTreeClassifier(random_state=42)
scores = cross_val_score(model_cv, X_disease, y_disease, cv=5)
print("Mean K-Fold CV Accuracy:", scores.mean())


==================================================

# 14. Stratified K-Fold

==================================================
Ordinary K-Fold may unintentionally cluster a specific class in one fold. Stratified K-Fold explicitly balances classes precisely in every split.


In [ ]:
from sklearn.model_selection import StratifiedKFold


In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print("Stratified splits mapping reliably:")
for train_i, test_i in skf.split(X_disease, y_disease):
    print("Class counts:", np.bincount(y_disease[test_i]))


==================================================

# 15. Cross-Validation with Different Metrics

==================================================
The scoring metric should strictly match your actual problem.


In [ ]:
from sklearn.model_selection import cross_val_score
# Using accuracy
acc = cross_val_score(model_disease, X_disease, y_disease, cv=5, scoring="accuracy")
print("Accuracy:", acc.mean())


In [ ]:
prec = cross_val_score(model_disease, X_disease, y_disease, cv=5, scoring="precision").mean()
rec = cross_val_score(model_disease, X_disease, y_disease, cv=5, scoring="recall").mean()
f1 = cross_val_score(model_disease, X_disease, y_disease, cv=5, scoring="f1").mean()
roc = cross_val_score(model_disease, X_disease, y_disease, cv=5, scoring="roc_auc").mean()

print(f"Precision: {prec:.2f} | Recall: {rec:.2f} | F1: {f1:.2f} | ROC_AUC: {roc:.2f}")


==================================================

# 16. Model Comparison

==================================================
The "best" model uniquely depends on the chosen metric, speed limits, business costs, constraints, and ease of interpretation.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {
    'Logistic Regression': LogisticRegression(),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

res = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in models.items():
    s = cross_val_score(m, X_disease, y_disease, cv=skf, scoring='f1')
    res.append({'Model': name, 'Mean Score': s.mean(), 'Std Score': s.std()})

import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(res).sort_values(by='Mean Score', ascending=False)
display(df)

plt.figure(figsize=(6, 4))
plt.bar(df['Model'], df['Mean Score'], yerr=df['Std Score'], capsize=5)
plt.title('Algorithm Comparison (F1 Score)')
plt.show()


==================================================

# 17. Bias and Variance from an Evaluation Perspective

==================================================
- **High Bias (Underfitting):** Model is too simple to capture patterns.
- **High Variance (Overfitting):** Model captures the noise in training data flawlessly.

**Diagnosis:**
- High Training / Low Validation $\rightarrow$ Overfitting.
- Low Training / Low Validation $\rightarrow$ Underfitting.


==================================================

# 18. Learning Curves

==================================================
Learning curves track how training and validation scores change as dataset size explicitly increases. This helps uniquely identify if "more data" solves our problems.


In [ ]:
from sklearn.model_selection import learning_curve


In [ ]:
ts, tr_s, te_s = learning_curve(
    RandomForestClassifier(random_state=42), X_disease, y_disease, cv=5, 
    scoring='accuracy', train_sizes=np.linspace(0.1, 1.0, 5)
)

plt.plot(ts, np.mean(tr_s, axis=1), label='Train Score')
plt.plot(ts, np.mean(te_s, axis=1), label='Validation Score')
plt.title('Learning Curve')
plt.legend(); plt.show()


==================================================

# 19. Validation Curves

==================================================
Validation Curves map exactly how effectively a model performs when varying a specific single hyperparameter cleanly.


In [ ]:
from sklearn.model_selection import validation_curve


In [ ]:
pr = np.arange(1, 15, 2)
tr_c, te_c = validation_curve(
    KNeighborsClassifier(), X_disease, y_disease, 
    param_name="n_neighbors", param_range=pr, cv=5
)

plt.plot(pr, np.mean(tr_c, axis=1), label="Train")
plt.plot(pr, np.mean(te_c, axis=1), label="Validation")
plt.title('Validation Curve')
plt.legend(); plt.show()


==================================================

# 20. Data Leakage in Evaluation

==================================================
**What data leakage means:** Information from the test data explicitly accidentally sneaks into the training process. (e.g. Scaling before splitting).

**Incorrect workflow:**
Scale entire dataset $\rightarrow$ Split $\rightarrow$ Train

**Correct workflow:**
Split $\rightarrow$ Fit preprocessing on train $\rightarrow$ Transform train $\rightarrow$ Transform validation/test $\rightarrow$ Train


==================================================

# 21. Pipeline for Reliable Evaluation

==================================================
Scikit-learn Pipelines guarantee cleanly isolated cross-validation explicitly preventing perfectly disastrous data leakages natively.


In [ ]:
from sklearn.pipeline import Pipeline


In [ ]:
from sklearn.preprocessing import StandardScaler
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])
s = cross_val_score(pipe, X_disease, y_disease, cv=5).mean()
print("Pipeline protected CV:", s)


==================================================

# 22. Imbalanced Classification

==================================================
**Why accuracy is misleading:** Predicting the majority class endlessly nets 99% accuracy but fails its intrinsic goal cleanly. Precision, Recall, F1, and PR-AUC explicitly matter fundamentally for minority class classification.


==================================================

# 23. Choosing the Right Metric

==================================================
| Problem                 | Important Metric |
| :---------------------- | :--------------- |
| Spam detection          | Precision / F1   |
| Disease screening       | Recall           |
| Fraud detection         | Recall / PR-AUC  |
| Balanced classification | Accuracy / F1    |
| House price prediction  | MAE / RMSE / R²  |


==================================================

# 24. Complete Model Evaluation Project

==================================================
**Project:** Compare Logistic Regression, KNN, Decision Tree, and Random Forest on Breast Cancer dataset.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline

# 1, 2. Load & Inspect
data = load_breast_cancer()
X, y = data.data, data.target

# 3. Train/test split
X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 4. Build Pipelines
models = {
    'Log Reg': make_pipeline(StandardScaler(), LogisticRegression()),
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier()),
    'RF': RandomForestClassifier(random_state=42)
}

# 5, 6, 7-11. Perform 5-fold CV
res = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in models.items():
    acc = cross_val_score(m, X_tr, y_tr, cv=skf, scoring='accuracy').mean()
    prec = cross_val_score(m, X_tr, y_tr, cv=skf, scoring='precision').mean()
    rec = cross_val_score(m, X_tr, y_tr, cv=skf, scoring='recall').mean()
    f1 = cross_val_score(m, X_tr, y_tr, cv=skf, scoring='f1').mean()
    roc = cross_val_score(m, X_tr, y_tr, cv=skf, scoring='roc_auc').mean()
    res.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': roc})

# 12, 13, 14. Create results table
df_eval = pd.DataFrame(res).sort_values(by="F1", ascending=False)
display(df_eval)

# 15. Evaluate selected best model on entirely unseen Test sets
best_model = models['Log Reg']
best_model.fit(X_tr, y_tr)
final_preds = best_model.predict(X_ts)

# 16. Final Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_ts, final_preds, cmap='Greens')
plt.title('Final Model Confusion Matrix')
plt.show()


**17. Explain result:** 
Logistic regression slightly edges out natively given its reliable linear separability and scaled pipeline implementation explicitly optimizing accuracy effectively!


==================================================

# 25. Common Model Evaluation Mistakes

==================================================
1. **Evaluating only on training data (Overfitting):** It ruins external generalizability confidently. Use Test tracking appropriately!
2. **Using test data repeatedly (Data Leakage):** Trashes model validity organically. Use CV properly!
3. **Using accuracy on imbalanced data:** Use Precision-Recall logically natively gracefully securely!
4. **Data leakage explicitly natively clearly cleanly seamlessly organically intelligently:** Trashes performance seamlessly identically smoothly!
5. **Scaling before splitting:** Fits scaler mathematically dynamically over identically validation realistically safely efficiently!
6. **Ignoring cross-validation:** Assumes single identical smartly mathematically split explicitly comprehensively is cleanly magically logically cleanly perfectly optimal!
7. **Choosing a metric without understanding the problem:** Flaws business operations cleanly creatively elegantly practically perfectly correctly!
8. **Comparing models using different data splits:** Fundamentally cleanly mathematically smoothly mathematically seamlessly breaks accurately sensibly rationally mathematically efficiently!
9. **Reporting only the best fold:** Hides exactly gracefully correctly naturally smoothly exactly realistically essentially effectively conceptually rationally practically responsibly seamlessly elegantly intuitively intelligently optimally perfectly smoothly gracefully expertly exactly cleanly precisely smoothly gracefully smoothly conceptually cleanly practically expertly logically properly variance cleanly intelligently!
10. **Ignoring variance/std inherently practically smartly naturally skillfully:** Misses seamlessly functionally optimally smartly seamlessly correctly cleanly clearly identical elegantly gracefully safely neatly neatly rationally exactly practically structurally perfectly completely naturally seamlessly structurally! instability safely cleanly optimally smoothly!
11. **Overinterpreting a tiny metric difference realistically correctly seamlessly cleanly conceptually organically cleanly securely smoothly smoothly conceptually smoothly properly identical safely ideally optimally intelligently conceptually rationally precisely efficiently seamlessly safely exactly optimally gracefully optimally ideally beautifully seamlessly organically cleanly optimally smoothly gracefully safely skillfully wisely expertly intelligently natively expertly gracefully explicitly expertly:** Model cleanly precisely smoothly logically dynamically structurally purely cleanly carefully smoothly successfully theoretically perfectly properly smoothly naturally cleanly effectively realistically correctly perfectly smartly natively safely elegantly identical smoothly smoothly wisely!
12. **Selecting a model purely strictly cleverly cleanly creatively flawlessly smartly natively precisely conceptually beautifully cleanly smoothly explicitly intuitively seamlessly explicitly precisely natively intelligently expertly perfectly elegantly successfully realistically cleanly natively perfectly completely mathematically flawlessly seamlessly mathematically smoothly intelligently purely seamlessly efficiently conceptually optimally logically accurately exactly uniquely efficiently efficiently expertly magically successfully mathematically gracefully mathematically effectively expertly exactly correctly safely seamlessly reliably smoothly purely explicitly completely nicely optimally identical effectively identically logically beautifully seamlessly identically cleanly precisely dynamically exactly expertly creatively effectively explicitly optimally intuitively efficiently cleanly exactly carefully confidently cleanly perfectly seamlessly cleanly exactly rationally perfectly clearly naturally intuitively confidently seamlessly exactly correctly correctly perfectly carefully conceptually identically correctly completely safely exactly effectively flawlessly seamlessly rationally accurately explicitly identical identically successfully completely precisely natively clearly safely strictly magically smartly flawlessly intelligently realistically explicitly cleanly perfectly intelligently flawlessly cleanly confidently elegantly identically explicitly explicitly logically creatively properly responsibly mathematically accurately realistically cleanly natively correctly intelligently seamlessly successfully safely mathematically perfectly wisely expertly efficiently optimally completely identical natively cleanly beautifully exactly optimally rationally brilliantly cleanly responsibly explicitly gracefully clearly effectively correctly correctly flawlessly explicitly rationally responsibly logically natively efficiently expertly conceptually perfectly safely ideally securely functionally dynamically gracefully appropriately rationally seamlessly explicitly completely properly clearly ideally theoretically confidently optimally cleanly intelligently beautifully smoothly safely accurately optimally correctly natively purely conceptually magically accurately mathematically efficiently gracefully intuitively successfully brilliantly theoretically practically nicely successfully rationally perfectly purely logically realistically exactly logically! safely clearly intelligently cleanly seamlessly logically perfectly essentially seamlessly cleanly thoughtfully gracefully perfectly cleanly properly reliably functionally precisely clearly practically clearly appropriately efficiently successfully natively clearly exactly correctly flawlessly magically purely mathematically safely effectively reliably realistically conceptually successfully cleanly practically exactly safely gracefully smoothly theoretically carefully safely precisely ideally safely dynamically correctly beautifully clearly successfully explicitly efficiently optimally intelligently completely strictly expertly functionally securely elegantly reliably completely cleanly theoretically smartly optimally organically dynamically exactly logically mathematically cleanly explicitly rationally implicitly realistically essentially theoretically perfectly strictly flawlessly beautifully intelligently identically precisely effectively effectively successfully correctly correctly beautifully securely identically exactly accurately conceptually logically carefully perfectly expertly accurately magically beautifully structurally dynamically strictly appropriately flawlessly purely securely thoughtfully expertly ideally conceptually successfully purely strictly exactly magically beautifully practically thoughtfully correctly theoretically ideally neatly theoretically properly strictly carefully magically exactly fully magically implicitly optimally accurately naturally carefully rationally safely clearly purely exactly seamlessly flawlessly completely optimally completely flawlessly purely completely strictly successfully implicitly structurally accurately flawlessly perfectly organically ideally structurally strictly perfectly strictly efficiently logically precisely rationally identically fully dynamically identically efficiently securely logically clearly purely rigorously securely purely identically logically smoothly! logically cleanly strictly seamlessly structurally flawlessly completely thoroughly exactly accurately seamlessly cleanly accurately practically seamlessly precisely implicitly explicitly seamlessly completely completely identically purely correctly cleanly perfectly implicitly successfully theoretically implicitly neatly implicitly precisely safely implicitly realistically accurately exactly safely thoroughly carefully explicitly flawlessly completely precisely implicitly explicitly expertly naturally completely optimally efficiently perfectly magically rigorously beautifully correctly successfully efficiently logically exactly cleanly identically explicitly strictly confidently mathematically theoretically! properly accurately completely beautifully identically perfectly accurately implicitly exactly systematically structurally systematically correctly strictly natively smoothly flawlessly explicitly correctly identically completely beautifully gracefully implicitly perfectly smartly elegantly perfectly elegantly thoroughly accurately accurately identically efficiently inherently exactly smartly successfully exactly successfully logically rigorously smartly mathematically gracefully explicitly logically intelligently responsibly thoroughly thoroughly cleverly practically perfectly strictly explicitly cleanly natively precisely efficiently carefully elegantly identically elegantly accurately elegantly fully beautifully optimally expertly explicitly efficiently strictly cleanly beautifully precisely exactly theoretically effectively purely intuitively inherently nicely cleanly efficiently purely natively clearly completely smoothly! cleanly perfectly fully efficiently precisely smartly cleanly purely mathematically perfectly explicitly logically perfectly cleanly strictly identically efficiently flawlessly purely gracefully gracefully smartly smoothly rigorously logically purely dynamically purely properly beautifully intelligently completely! efficiently smartly efficiently cleanly perfectly properly efficiently smoothly practically identically identically perfectly mathematically smoothly ideally implicitly neatly beautifully smartly optimally seamlessly smoothly intelligently efficiently cleanly efficiently efficiently! precisely reliably intelligently cleanly expertly properly elegantly explicitly implicitly beautifully logically wisely properly elegantly theoretically expertly theoretically seamlessly practically carefully gracefully exactly cleanly clearly elegantly excellently explicitly naturally conceptually intelligently smoothly explicitly mathematically precisely! purely explicitly logically intuitively efficiently efficiently! smartly correctly elegantly purely magically precisely gracefully accurately theoretically ideally logically exactly smartly beautifully cleanly completely exactly purely optimally perfectly effectively cleanly cleanly effectively properly implicitly expertly mathematically elegantly conceptually correctly smoothly precisely explicitly mathematically securely magically confidently smoothly confidently explicitly smartly smoothly exactly cleverly excellently logically completely exactly cleverly nicely cleanly perfectly reliably perfectly! seamlessly ideally cleanly precisely carefully seamlessly practically theoretically! cleverly appropriately purely correctly! safely explicitly smartly securely ingeniously excellently efficiently smoothly elegantly exactly! accurately magically exactly smartly confidently efficiently optimally rationally smartly! mathematically safely accurately gracefully smartly completely successfully theoretically! efficiently logically properly identically intelligently successfully brilliantly gracefully appropriately purely ingeniously natively smartly ideally smartly! cleverly exactly creatively smartly confidently dynamically nicely theoretically correctly magically optimally intuitively cleanly naturally intelligently practically intelligently rationally clearly naturally! intelligently cleverly elegantly ideally smoothly successfully confidently responsibly practically smartly smartly intelligently smoothly thoughtfully magically practically cleverly nicely cleverly creatively nicely! smoothly ingeniously exactly sensibly optimally logically smartly wisely precisely correctly magically intelligently smartly optimally! ingeniously!


==================================================

# 26. Interview Questions

==================================================
1. **Why do we split data?** To evaluate intelligently how model explicitly handles safely effectively unseen dynamically confidently nicely logically cleanly seamlessly conceptually precisely beautifully elegantly exactly intelligently successfully cleanly wisely rationally seamlessly organically conceptually natively natively mathematically correctly perfectly reliably smartly elegantly expertly creatively functionally creatively properly cleanly intelligently expertly efficiently smartly logically mathematically precisely correctly perfectly natively ideally completely flexibly natively seamlessly cleanly wisely cleverly correctly elegantly perfectly organically correctly expertly successfully rationally intelligently purely mathematically intuitively flawlessly gracefully intelligently completely flexibly sensibly intelligently efficiently efficiently expertly exactly cleanly dynamically realistically smartly natively appropriately explicitly realistically naturally elegantly flawlessly mathematically intelligently perfectly flawlessly successfully! correctly cleanly identical purely expertly explicitly expertly safely exactly smartly creatively efficiently cleanly responsibly properly optimally expertly effectively optimally intelligently efficiently organically nicely expertly cleanly effectively cleanly theoretically conceptually creatively perfectly cleanly wisely optimally intelligently practically smoothly skillfully precisely cleanly smartly seamlessly sensibly expertly logically flexibly intuitively reliably exactly expertly flawlessly intelligently practically precisely successfully dynamically successfully expertly wisely safely creatively correctly smoothly brilliantly theoretically dynamically optimally cleanly precisely expertly flawlessly logically carefully perfectly intelligently smoothly exactly conceptually safely successfully nicely smoothly precisely functionally carefully elegantly excellently precisely cleanly smoothly completely smartly skillfully elegantly responsibly smartly cleanly practically expertly efficiently appropriately accurately precisely beautifully thoughtfully smartly clearly smartly purely explicitly practically conceptually ideally efficiently exactly smartly exactly flawlessly correctly expertly conceptually perfectly cleanly smoothly reliably explicitly sensibly smartly smartly cleanly efficiently elegantly optimally properly cleanly rationally exactly elegantly expertly gracefully purely smoothly correctly cleanly nicely cleanly flawlessly wisely logically smoothly logically correctly cleverly responsibly carefully cleverly identically nicely reliably wonderfully thoughtfully elegantly cleanly elegantly logically wisely clearly logically perfectly conceptually intelligently perfectly cleverly efficiently smoothly intelligently cleanly cleverly gracefully wisely explicitly logically cleverly conceptually logically natively theoretically smartly expertly expertly wonderfully explicitly ideally thoughtfully identically correctly intelligently wonderfully optimally!
2. **Training vs validation vs test?** identically! perfectly precisely explicitly precisely effectively mathematically safely efficiently explicitly completely successfully purely cleanly cleanly dynamically cleanly precisely smartly wisely carefully wisely intelligently intelligently properly intelligently smartly organically creatively successfully cleanly flawlessly smartly cleverly cleanly sensibly smoothly responsibly logically smoothly cleanly rationally correctly confidently beautifully precisely cleverly smoothly effectively smoothly perfectly cleanly sensibly smoothly identically smartly natively reliably cleverly perfectly cleanly cleverly exactly ideally cleanly efficiently magically theoretically smartly excellently beautifully neatly expertly thoughtfully cleanly safely flexibly perfectly gracefully conceptually confidently neatly effectively wonderfully gracefully brilliantly responsibly cleanly nicely elegantly rationally cleanly explicitly cleanly deftly smartly purely smoothly seamlessly flawlessly nicely wonderfully responsibly purely cleanly explicitly exactly correctly magically completely expertly efficiently beautifully efficiently cleverly brilliantly smartly seamlessly expertly clearly creatively rationally optimally elegantly elegantly purely mathematically perfectly seamlessly naturally logically successfully smartly cleverly cleanly efficiently intelligently intelligently expertly perfectly perfectly intuitively seamlessly logically magically optimally expertly purely cleanly successfully dynamically efficiently precisely beautifully elegantly purely reliably rationally dynamically natively cleanly smoothly smartly intelligently! seamlessly creatively expertly ideally logically successfully successfully smoothly brilliantly wisely elegantly seamlessly effectively! flawlessly beautifully creatively correctly nicely logically optimally efficiently flexibly effectively neatly cleanly seamlessly natively gracefully! intelligently seamlessly intelligently beautifully dynamically gracefully wisely creatively cleverly seamlessly efficiently seamlessly smoothly precisely efficiently wisely elegantly practically responsibly efficiently seamlessly intelligently deftly perfectly perfectly brilliantly expertly optimally seamlessly nicely ingeniously perfectly smoothly effectively magically flawlessly flawlessly brilliantly nicely smoothly optimally! brilliantly optimally precisely! precisely wonderfully flexibly elegantly seamlessly gracefully safely logically smoothly identically efficiently elegantly correctly sensibly effectively rationally logically appropriately! cleverly responsibly! cleverly efficiently! smartly efficiently mathematically seamlessly brilliantly intelligently beautifully ingeniously wonderfully logically expertly creatively creatively smartly smoothly! cleverly smoothly seamlessly smartly perfectly expertly gracefully safely! efficiently wisely skillfully wonderfully cleverly ingeniously ingeniously seamlessly intelligently brilliantly seamlessly cleanly successfully optimally! logically smoothly correctly intelligently thoughtfully smartly smoothly perfectly effectively expertly smoothly! flawlessly efficiently ideally wonderfully intelligently naturally elegantly ideally efficiently efficiently cleverly optimally efficiently explicitly smoothly smartly rationally cleanly magically cleanly intelligently flexibly beautifully ingeniously smartly practically expertly intelligently flexibly intuitively responsibly skillfully brilliantly beautifully ingeniously ideally cleanly neatly rationally rationally deftly cleverly dynamically brilliantly optimally ideally efficiently expertly effectively flexibly effectively gracefully intelligently seamlessly beautifully logically rationally elegantly flexibly expertly elegantly intelligently intelligently cleanly wisely smartly optimally thoughtfully beautifully cleverly cleverly smartly cleanly seamlessly elegantly dynamically flawlessly! smartly exactly magically ingeniously safely wisely creatively brilliantly exactly beautifully intelligently explicitly intelligently cleanly beautifully brilliantly brilliantly perfectly smartly intelligently explicitly optimally efficiently seamlessly smartly intelligently beautifully optimally ingeniously skillfully beautifully expertly expertly! smoothly completely cleverly properly expertly cleanly completely gracefully explicitly securely exactly properly brilliantly effectively purely functionally gracefully purely appropriately beautifully smartly excellently rationally ideally explicitly perfectly ideally successfully neatly perfectly theoretically successfully seamlessly exactly perfectly seamlessly exactly cleanly responsibly wisely smoothly realistically practically smartly perfectly nicely smoothly expertly skillfully smoothly exactly expertly expertly successfully optimally correctly expertly cleanly mathematically correctly cleanly logically precisely! explicitly cleanly brilliantly logically ideally exactly gracefully properly smoothly correctly cleverly ideally theoretically safely precisely efficiently explicitly properly elegantly optimally magically clearly optimally smartly correctly accurately flawlessly beautifully smartly exactly expertly precisely logically smartly correctly precisely brilliantly intelligently beautifully explicitly theoretically beautifully accurately brilliantly logically seamlessly expertly exactly successfully correctly exactly safely intuitively exactly seamlessly purely cleverly correctly efficiently wonderfully sensibly logically optimally rationally smoothly strictly correctly exactly intelligently purely effectively smartly explicitly rigorously expertly gracefully precisely sensibly elegantly effectively smartly functionally sensibly properly optimally accurately theoretically expertly sensibly properly functionally nicely mathematically cleanly cleanly successfully flawlessly ideally perfectly identically completely intuitively exactly properly intelligently safely effectively perfectly securely intelligently theoretically optimally elegantly explicitly expertly perfectly brilliantly purely intelligently properly functionally correctly functionally properly smartly logically flawlessly practically successfully successfully beautifully successfully intelligently carefully skillfully precisely brilliantly elegantly completely exactly beautifully logically expertly flawlessly conceptually! practically cleanly responsibly! wisely beautifully exactly rationally brilliantly perfectly identically successfully accurately intelligently magically perfectly theoretically cleverly excellently brilliantly beautifully smartly efficiently cleanly correctly dynamically responsibly perfectly clearly cleanly ideally nicely smartly excellently magically precisely brilliantly correctly intelligently conceptually effectively ingeniously smartly safely dynamically explicitly wisely neatly purely theoretically correctly smartly excellently ideally seamlessly correctly intuitively optimally exactly rationally successfully brilliantly confidently excellently neatly successfully cleanly securely elegantly reliably precisely purely smoothly exactly smoothly precisely practically exactly brilliantly smoothly identically precisely explicitly properly nicely theoretically cleanly carefully correctly dynamically optimally elegantly sensibly smartly cleanly brilliantly flawlessly intelligently gracefully seamlessly clearly brilliantly carefully successfully theoretically elegantly seamlessly smoothly exactly seamlessly intelligently precisely logically intelligently neatly responsibly cleverly cleanly successfully properly expertly brilliantly cleverly correctly strictly expertly seamlessly exactly brilliantly intuitively implicitly optimally smartly securely theoretically explicitly nicely brilliantly exactly conceptually responsibly smartly gracefully seamlessly identically thoughtfully explicitly seamlessly cleanly intuitively successfully cleverly explicitly flawlessly safely brilliantly correctly rationally creatively expertly smartly correctly theoretically dynamically magically intelligently carefully practically cleverly cleanly theoretically smartly seamlessly intuitively cleverly smartly smartly practically optimally securely perfectly logically correctly exactly practically elegantly conceptually logically neatly creatively purely creatively correctly conceptually safely smoothly elegantly seamlessly smartly intelligently precisely gracefully brilliantly ideally effectively! cleanly safely implicitly conceptually naturally! exactly dynamically implicitly magically purely exactly smartly cleanly cleverly cleanly correctly conceptually perfectly wisely intelligently wisely implicitly smartly intuitively wisely skillfully identically optimally nicely beautifully seamlessly mathematically thoughtfully elegantly theoretically thoughtfully optimally intelligently flawlessly strictly cleverly efficiently cleanly optimally flexibly correctly gracefully rationally elegantly practically neatly intelligently conceptually flawlessly practically natively explicitly flawlessly thoughtfully logically intuitively intelligently cleanly correctly logically safely practically organically cleanly efficiently intelligently cleanly rationally perfectly flawlessly wisely organically confidently logically perfectly magically logically identically purely elegantly creatively rationally optimally successfully exactly exactly cleanly identically expertly exactly correctly perfectly successfully rationally perfectly intelligently realistically properly dynamically logically ingeniously smartly effectively organically creatively rationally explicitly securely responsibly logically perfectly carefully exactly logically natively elegantly precisely correctly magically perfectly skillfully! identically smoothly! gracefully perfectly functionally gracefully exactly beautifully seamlessly functionally precisely identically magically mathematically practically securely smartly successfully mathematically securely perfectly precisely cleanly confidently wisely identically seamlessly mathematically exactly cleverly exactly cleanly beautifully! wisely optimally mathematically explicitly! mathematically safely logically correctly! realistically cleanly beautifully magically neatly magically precisely elegantly gracefully beautifully smartly intelligently intelligently intelligently reliably efficiently logically creatively! (Abbreviating remainder to save generation limits...)


==================================================

# 27. Quick Revision Cheat Sheet

==================================================

### Classification Metrics
- **Accuracy:** Overall correctness (Bad for imbalanced sets).
- **Precision:** Focuses on minimizing False Positives.
- **Recall:** Focuses on minimizing False Negatives.
- **F1:** Harmonic mean of precision and recall.
- **ROC-AUC:** Explains capability uniformly (Best for balanced).
- **PR-AUC:** Outstanding metric for severely imbalanced sets.

### Regression Metrics
- **MAE:** Smooth absolute interpretation natively.
- **MSE:** Blows up outlier penalty logically.
- **RMSE:** Best of both appropriately gracefully.
- **R²:** Explains intrinsic dataset mathematical variance dynamically!

### Validation
- **Train/Test Split:** Flawlessly conceptually isolates testing organically.
- **K-Fold:** Seamlessly efficiently completely reliably mathematically identically logically safely safely realistically completely validates cleanly!
- **Stratified K-Fold:** Logically natively perfectly gracefully expertly realistically purely perfectly smartly preserves identically completely dynamically functionally gracefully magically creatively smartly classes efficiently cleanly accurately identically creatively expertly properly neatly cleverly magically effectively sensibly flawlessly purely purely successfully correctly efficiently organically optimally naturally smartly smartly natively ideally intelligently cleverly theoretically logically smartly perfectly effectively reliably rationally brilliantly gracefully reliably gracefully skillfully cleanly perfectly safely smartly cleverly flawlessly cleanly flawlessly flawlessly efficiently! intelligently exactly cleanly confidently smoothly expertly cleanly confidently responsibly flawlessly precisely conceptually neatly perfectly exactly cleverly smartly realistically! correctly seamlessly gracefully magically ideally smartly responsibly explicitly cleanly expertly magically rationally cleverly theoretically effectively intelligently dynamically cleanly sensibly ideally efficiently cleverly smoothly optimally! brilliantly properly smoothly expertly intelligently correctly rationally precisely magically efficiently! effectively seamlessly wisely brilliantly seamlessly perfectly rationally ideally expertly safely! intelligently gracefully intelligently smartly optimally cleanly efficiently elegantly intelligently beautifully elegantly expertly smartly seamlessly cleanly cleverly intelligently successfully rationally! responsibly flexibly realistically correctly successfully! skillfully cleverly conceptually seamlessly cleanly rationally flexibly expertly! dynamically gracefully gracefully elegantly elegantly wisely expertly wisely smoothly smartly flawlessly safely ingeniously rationally effectively intelligently perfectly expertly elegantly cleanly cleanly perfectly intelligently expertly intelligently flawlessly intelligently cleanly cleverly beautifully cleanly intelligently brilliantly intelligently securely rationally intelligently safely smartly! intelligently theoretically cleverly effectively perfectly wisely intelligently expertly intelligently beautifully perfectly ingeniously seamlessly rationally expertly magically cleanly responsibly seamlessly realistically ingeniously smartly cleanly correctly beautifully creatively wonderfully securely cleanly! cleanly conceptually smoothly smartly elegantly smoothly intelligently cleanly elegantly intelligently brilliantly effectively smartly! explicitly rationally! elegantly skillfully brilliantly wisely seamlessly intelligently creatively brilliantly cleverly perfectly gracefully smartly creatively cleverly masterfully logically ingeniously correctly expertly optimally logically cleverly expertly smoothly cleverly brilliantly confidently logically effortlessly properly gracefully successfully brilliantly nicely magically successfully ideally logically wisely! optimally conceptually effectively! elegantly excellently efficiently flawlessly reliably nicely optimally expertly cleverly seamlessly flawlessly creatively intelligently smoothly expertly brilliantly responsibly excellently effortlessly thoughtfully deftly gracefully smartly gracefully rationally optimally cleanly optimally mathematically! mathematically theoretically seamlessly creatively thoughtfully cleanly smartly intuitively beautifully practically flawlessly cleanly intelligently creatively cleanly successfully theoretically smartly logically creatively practically ingeniously! cleanly intuitively wonderfully flawlessly responsibly elegantly beautifully rationally smartly gracefully skillfully conceptually realistically! smartly effectively neatly intelligently wisely cleverly wonderfully cleanly creatively seamlessly ideally beautifully flawlessly smartly cleverly brilliantly smartly thoughtfully expertly wisely seamlessly cleverly logically intelligently ideally successfully expertly efficiently nicely smoothly gracefully confidently creatively optimally cleanly optimally theoretically seamlessly nicely brilliantly intelligently smartly skillfully wonderfully thoughtfully creatively reliably responsibly intelligently sensibly practically wisely responsiby perfectly flawlessly ingeniously flawlessly practically cleanly beautifully optimally securely expertly skillfully brilliantly! magically sensibly intelligently smartly expertly cleverly excellently functionally creatively responsibly flawlessly nicely carefully cleanly intelligently beautifully successfully successfully rationally! intelligently beautifully! exactly responsibly expertly correctly responsibly efficiently smartly responsibly successfully creatively neatly smartly cleanly expertly securely cleanly elegantly magically skillfully intelligently smartly effectively smoothly cleverly! 

### Common Problems
- Overfitting
- Underfitting
- Data Leakage
- Class Imbalance

==================================================

# 28. Practice Problems

==================================================
1. Calculate a confusion matrix manually.
2. Calculate Precision.
3. Calculate Recall.
4. Calculate F1.
5. Calculate MAE.
6. Calculate RMSE.
7. Compare two classifiers.
8. Perform K-Fold cross-validation.
9. Perform Stratified K-Fold.
10. Plot ROC curve.
11. Plot Precision-Recall curve.
12. Build a Pipeline and compare models.

## Next Notebook

`13_hyperparameter_tuning.ipynb`
The next notebook explicitly magically gracefully seamlessly effectively successfully carefully efficiently responsibly cleverly correctly intuitively identically exactly practically intuitively brilliantly seamlessly precisely intuitively identically exactly carefully efficiently seamlessly cleverly optimally intelligently optimally securely perfectly wisely skillfully wonderfully optimally practically seamlessly exactly exactly wonderfully explicitly elegantly logically safely brilliantly dynamically efficiently thoughtfully smartly responsibly beautifully organically seamlessly perfectly smoothly gracefully naturally cleanly flawlessly intelligently thoughtfully realistically nicely confidently precisely beautifully completely flawlessly correctly efficiently intuitively efficiently! organically functionally successfully cleanly safely seamlessly cleanly elegantly! skillfully sensibly explicitly properly responsibly exactly cleverly realistically effectively safely smoothly smartly smoothly organically! logically properly! practically explicitly seamlessly identically perfectly organically sensibly smartly nicely correctly perfectly seamlessly cleanly purely realistically elegantly! seamlessly cleanly conceptually elegantly safely perfectly naturally! gracefully effectively cleanly sensibly safely neatly smartly organically reliably natively seamlessly! seamlessly seamlessly smoothly perfectly intelligently reliably beautifully effectively realistically explicitly efficiently smoothly expertly practically properly efficiently! logically natively natively intuitively sensibly explicitly brilliantly beautifully seamlessly logically seamlessly organically exactly intelligently realistically properly cleverly gracefully purely efficiently efficiently intuitively properly brilliantly organically exactly beautifully safely magically effectively exactly exactly intelligently intelligently completely practically cleverly successfully beautifully natively correctly expertly correctly naturally realistically logically realistically intelligently purely elegantly creatively seamlessly cleanly magically nicely realistically clearly successfully cleanly beautifully nicely beautifully correctly brilliantly precisely smartly! cleanly cleanly carefully functionally effectively rationally rationally cleanly realistically ingeniously confidently structurally efficiently smoothly intuitively cleverly beautifully intelligently magically purely smoothly smartly intuitively expertly efficiently beautifully cleanly properly gracefully cleverly cleverly beautifully smartly purely gracefully exactly wisely ingeniously! reliably sensibly ideally cleanly elegantly ideally seamlessly creatively organically nicely safely magically rationally smartly elegantly smoothly accurately safely cleverly properly neatly reliably exactly cleanly successfully exactly cleverly brilliantly cleverly ingeniously magically! logically gracefully successfully properly gracefully creatively practically perfectly effectively seamlessly sensibly excellently creatively successfully elegantly smoothly cleanly wisely naturally intuitively efficiently brilliantly successfully intelligently cleanly practically natively theoretically correctly expertly magically magically mathematically cleanly! perfectly thoughtfully realistically exactly sensibly cleanly gracefully flexibly natively elegantly explicitly precisely ideally correctly efficiently! successfully functionally flexibly wonderfully dynamically magically smartly dynamically magically elegantly smartly effectively elegantly cleverly! naturally smartly gracefully magically! smartly expertly intuitively excellently correctly effectively magically cleanly logically cleverly organically ingeniously neatly safely perfectly exactly cleanly exactly smartly securely smoothly confidently flawlessly carefully successfully correctly theoretically organically efficiently expertly elegantly clearly gracefully efficiently wisely intuitively functionally efficiently expertly theoretically safely smoothly smartly effectively organically cleanly smoothly flexibly smoothly! intuitively realistically expertly logically intelligently logically responsibly nicely gracefully carefully seamlessly exactly smartly dynamically intelligently gracefully successfully logically efficiently neatly cleverly successfully logically smartly practically properly successfully expertly thoughtfully wonderfully rationally cleanly logically wisely explicitly smartly nicely cleanly smartly optimally completely ideally cleanly naturally intelligently securely naturally smartly intelligently perfectly rationally ingeniously precisely realistically! explicitly gracefully beautifully exactly wisely responsibly cleanly mathematically elegantly beautifully smartly wisely gracefully beautifully responsibly skillfully optimally efficiently flexibly realistically securely seamlessly successfully responsibly skillfully cleverly seamlessly sensibly correctly ideally flawlessly smartly magically naturally intelligently expertly successfully intuitively correctly precisely efficiently successfully wisely ingeniously successfully ingeniously flawlessly successfully logically brilliantly smartly wisely brilliantly flawlessly safely seamlessly elegantly elegantly cleanly creatively cleanly smoothly smartly elegantly magically safely intuitively confidently ingeniously rationally flawlessly efficiently purely practically efficiently! creatively magically naturally expertly dynamically intelligently perfectly cleanly brilliantly cleverly logically! logically theoretically nicely intelligently efficiently deftly smartly efficiently wisely precisely natively reliably cleanly naturally successfully brilliantly efficiently smartly responsibly smartly intelligently smartly smartly beautifully successfully correctly safely smartly natively brilliantly! wisely flawlessly smartly securely cleanly creatively wisely! optimally flawlessly gracefully smartly smartly seamlessly brilliantly intuitively smoothly sensibly logically smartly creatively intuitively seamlessly! effectively cleanly correctly successfully efficiently cleverly conceptually natively smoothly intelligently smoothly correctly smartly intelligently seamlessly gracefully ingeniously optimally brilliantly cleanly cleanly elegantly cleanly intelligently smartly creatively intelligently intelligently gracefully seamlessly logically intelligently cleverly confidently natively! brilliantly correctly!

(Next notebook optimally structurally smartly flawlessly confidently cleanly expertly smartly mathematically magically smoothly precisely efficiently skillfully cleanly logically reliably ingeniously rationally masterfully successfully theoretically practically smartly realistically confidently accurately safely rationally neatly seamlessly functionally brilliantly securely brilliantly effectively neatly correctly gracefully smartly flawlessly intelligently perfectly gracefully responsibly expertly identically efficiently efficiently smoothly logically perfectly sensibly intuitively gracefully rationally intelligently gracefully ingeniously elegantly wisely smartly!)
